In [ ]:
import json
import numpy as np
from src.utils import parse_raw, simple_line_plot, fourier_transform_plot, segment_data, segment_plot
import plotly.express as px
import pandas as pd
import plotly.graph_objects as go
import os
from glob import glob

In [ ]:
raw_data_path = "raw_data/Viviane/S3/*"

all_files = glob(raw_data_path)


,timestamp,THORAX,X,Y
0,2025-09-19 09:48:52.424,-1.446533,-0.101250,0.880930
1,2025-09-19 09:48:52.429,-1.437378,-0.101875,0.877829
2,2025-09-19 09:48:52.434,-1.434326,-0.103437,0.884031
3,2025-09-19 09:48:52.439,-1.416016,-0.105000,0.884031
4,2025-09-19 09:48:52.444,-1.408386,-0.102500,0.877829


In [ ]:

file_path = "raw_data/protocole_respiration_viviane_induct.txt"
metadata_path = "raw_data/protocole_respiration_viviane_induct.json"

with open(file_path, 'r') as file:
    file_content = file.readlines()
    
header_json, df = parse_raw(file_content)

if os.path.exists(metadata_path):
    with open(metadata_path, 'r') as meta_file:
        metadata = json.load(meta_file)

df.head()

In [21]:
fig = simple_line_plot(df)
fig

In [22]:
fig = fourier_transform_plot(df)
fig

In [23]:
autocorr_thorax = np.correlate(df['THORAX'], df['THORAX'], mode='full')
lags = np.arange(-len(df['THORAX']) + 1, len(df['THORAX']))

fig = px.line(x=lags, y=autocorr_thorax, title='Autocorrelation of THORAX')
fig.update_xaxes(title_text='Lag')
fig.update_yaxes(title_text='Autocorrelation')
fig

In [24]:
autocorr_X = np.correlate(df['X'], df['X'], mode='full')
lags = np.arange(-len(df['X']) + 1, len(df['X']))

fig = px.line(x=lags, y=autocorr_X, title='Autocorrelation of X')
fig.update_xaxes(title_text='Lag')
fig.update_yaxes(title_text='Autocorrelation')
fig

In [28]:
def energie_moyenne(df, column : str ="X"):
    # Find the closest dip to 0 in autocorr_X (excluding the central peak)
    mid = len(autocorr_X) // 2
    # Search for local minima around the center
    search_range = autocorr_X[mid-500:mid+500]
    dip_idx = np.argmin(search_range)
    closest_dip = dip_idx - 500  # relative to center
    window_size = 10 * abs(closest_dip)
    energies = df[column].rolling(window=window_size, min_periods=1).apply(lambda x: np.mean(x**2), raw=True)
    return energies

fig = px.line(energie_moyenne(df,"X"))
fig

In [26]:
print(json.dumps(metadata, indent=4))

{
    "studentId": "68411A",
    "sequenceDescription": "Marche sur toute la longueur de la rue",
    "sessionId": "S3",
    "sequences": [
        {
            "sequenceId": 1,
            "begin": 0,
            "end": 6265,
            "sequenceContext": "REPOS"
        },
        {
            "sequenceId": 2,
            "begin": 6265,
            "end": 9880,
            "sequenceContext": "APNEE"
        },
        {
            "sequenceId": 3,
            "begin": 10325,
            "end": 17350,
            "sequenceContext": "MARCHE"
        },
        {
            "sequenceId": 4,
            "begin": 17351,
            "end": 19280,
            "sequenceContext": "REPOS"
        },
        {
            "sequenceId": 5,
            "begin": 19281,
            "end": 33029,
            "sequenceContext": "MONTEE"
        },
        {
            "sequenceId": 6,
            "begin": 33030,
            "end": 40048,
            "sequenceContext": "REPOS"
        }
    ]
}


In [27]:
fig = segment_plot(df, metadata["sequences"])
fig